**LaBr3：事件构建修复与回归记录**

本笔记记录 2026-09-27 完成的 B-01、B-03 和 P0-01：
保存小样本基线，明确正确事件分组，并修复三套程序中的末事件遗漏和多重数越界。

阅读顺序是项目流程、问题与规则、修复前后对比、异常处理、完整回归。
代码直接调用项目的 `tests/`，运行实际 C++ 程序，输出可通过“重启内核并运行全部单元”重新生成。

项目包括 `DT5720_DPP_PSD`、`DT5730_DPP_PHA`、`DT5730_DPP_PSD` 三套实现。

| 阶段 | 输入与输出 | 本次关联 |
| --- | --- | --- |
| `raw2root` | BIN → 按通道保存的 ROOT 树 | 保留已有文件结束处理的回归检查 |
| `alignment` | 通道时间戳 → 时间差直方图 | 辅助确定时间偏移，后续完善 |
| `sort` | 通道树与参数 → 按时间排序的单条记录 | 为 `build` 提供 `board/channel/energy/timestamp` |
| `build` | 排序记录 → 含 `m` 和数组分支的事件树 | 本次修复的核心 |
| `analysis` | 事件树 → 时间差分布 | 筛选和读取容量问题列入下一任务 |

详细配置见 [README](../README.md)，后续工作见 [PLAN](../PLAN.md)。

运行需要 Python 3.8 或更新版本、可导入的 PyROOT、`root-config`、GNU Make、C++ 编译器和 Git。
在 Jupyter 中选择能够执行 `import ROOT` 的 Python 内核，从上到下运行。
编译和测试可能需要一分钟左右，具体取决于机器。

默认从当前目录向上定位项目；在其他位置启动内核时，可在下一单元设置 `PROJECT_DIR`。
旧版对比需要本地 Git 历史中保留 `b096cd8`。
Notebook 使用临时合成输入和临时编译产物，结束后自动清理。

In [1]:
from pathlib import Path
import hashlib
import io
import platform
import shutil
import subprocess
import sys
import tempfile
import unittest
from datetime import datetime, timezone
from html import escape

PROJECT_DIR = None  # 也可设为项目根目录字符串
BASELINE_REV = "b096cd8"
DEMO_VARIANT = "DT5730_DPP_PSD"

if PROJECT_DIR is None:
    REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "tests/test_build.py").is_file()
                 and (p / "DT5730_DPP_PSD/build/Build.cpp").is_file()), None)
    if REPO is None:
        raise RuntimeError("无法定位项目，请设置 PROJECT_DIR。")
else:
    REPO = Path(PROJECT_DIR).expanduser().resolve()
    if not (REPO / "tests/test_build.py").is_file():
        raise RuntimeError("PROJECT_DIR 不是包含 tests/ 的 LaBr3 项目目录。")

for executable in ("root-config", "make", "g++", "git"):
    if shutil.which(executable) is None:
        raise RuntimeError(f"缺少命令：{executable}")

sys.path.insert(0, str(REPO / "tests"))
from fixtures import Hit, VARIANTS, checked_run, hits_at, read_events, write_sorted

import ROOT
ROOT.gROOT.SetBatch(True)

watched_files = [REPO / variant / "build" / name
                 for variant in VARIANTS
                 for name in ("Build.cpp", "Build.h", "main.cpp", "set.h")]
watched_files += [REPO / variant / "raw2root/set.h" for variant in VARIANTS]
source_hashes_before = {str(p.relative_to(REPO)): hashlib.sha256(p.read_bytes()).hexdigest()
                        for p in watched_files}

print("执行时间（UTC）：", datetime.now(timezone.utc).isoformat(timespec="seconds"))
print("Python：", platform.python_version())
print("ROOT：", checked_run(["root-config", "--version"]).stdout.strip())
print("当前 HEAD：", checked_run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO).stdout.strip())
print("对比基线：", BASELINE_REV)
print("新版程序使用当前工作区源码；HEAD 编号不包含尚未提交的修改。")


执行时间（UTC）： 2026-09-27T04:07:56+00:00
Python： 3.14.4
ROOT： 6.40.04
当前 HEAD： b096cd8
对比基线： b096cd8
新版程序使用当前工作区源码；HEAD 编号不包含尚未提交的修改。


In [2]:
class ResultTable:
    def __init__(self, headers, rows):
        self.headers = headers
        self.rows = rows

    def __repr__(self):
        lines = [" | ".join(map(str, self.headers))]
        lines.extend(" | ".join(map(str, row)) for row in self.rows)
        return "\n".join(lines)

    def _repr_html_(self):
        header = "".join(f"<th>{escape(str(value))}</th>" for value in self.headers)
        rows = "".join("<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row)
                       + "</tr>" for row in self.rows)
        return f"<table><thead><tr>{header}</tr></thead><tbody>{rows}</tbody></table>"


ResultTable(["设备配置", "当前 Build.cpp 的 SHA-256 前 12 位"],
            [[variant, source_hashes_before[f"{variant}/build/Build.cpp"][:12]]
             for variant in VARIANTS])


设备配置,当前 Build.cpp 的 SHA-256 前 12 位
DT5720_DPP_PSD,0c0dc04e272a
DT5730_DPP_PHA,7cc06ffb1e7f
DT5730_DPP_PSD,647b25571463


旧版 `Build::Process()` 只有在遇到窗口外的下一条记录时才调用 `Fill()`。
因此，文件结束时仍留在内存中的最后一个事件不会写入；单记录输入甚至得到零事件。

原来的容量条件为 `ev.m > MAX_MULTI`，检查太晚，不能保护第 9 条同窗记录写入长度为 8 的数组。
此外，空输入和时间戳倒退需要明确处理。

此前在三套旧程序上运行单记录、末尾符合事件和窗口边界三项检查，共复现 9 次失败。
下面对 `DT5730_DPP_PSD` 实际编译新旧两个版本，演示末事件遗漏；最后再对三套配置执行完整回归。

本次保留原有分组定义：时间戳单位为 **ps**，以事件首条记录为基准，
满足 `0 <= t - t_first < 50_000 ps` 的记录进入同一事件。

| 输入时间（ns） | 预期分组 | 用途 |
| --- | --- | --- |
| `[1]` | `[[1]]` | 单记录也必须输出 |
| `[0, 1, 80, 90]` | `[[0, 1], [80, 90]]` | 最后一个符合事件必须保留 |
| `[0, 49.999, 50, 50.001]` | `[[0, 49.999], [50, 50.001]]` | 恰好 50 ns 开启新事件 |
| `[0, 40, 80]` | `[[0, 40], [80]]` | 窗口以首记录为基准 |

多重数上限为 8。超出容量和时间戳倒退会明确失败，不静默改变分组或舍弃记录。

In [3]:
example_hits = hits_at([0, 49999, 50000, 50001])
ResultTable(["记录序号", "board", "channel", "energy", "timestamp / ps", "time / ns"],
            [[i, hit.board, hit.channel, hit.energy, hit.timestamp, hit.timestamp / 1000]
             for i, hit in enumerate(example_hits)])


记录序号,board,channel,energy,timestamp / ps,time / ns
0,1,0,0.125,0,0.0
1,2,1,1.125,49999,49.999
2,3,2,2.125,50000,50.0
3,4,3,3.125,50001,50.001


修复同步应用于三套 `build/Build.cpp`、`Build.h` 和 `main.cpp`：

1. 从第 0 条记录正常迭代，空树保持零事件；循环结束后补写非空末事件。
2. 先判断是否开启新窗口，再在追加前检查 `ev.m >= MAX_MULTI`，使“8 重后开启新事件”仍正常。
3. 检查时间戳顺序，用整数 ps 比较窗口，并处理有符号时间戳的极端跨度。
4. 检查输入文件、树、标量分支类型和读取结果；通过异常和非零退出码报告失败。
5. 先写同目录下的临时文件，完整写入并关闭后才改名为最终输出。失败时清理临时文件，保留已有输出。

代码示例见 [PSD Build.cpp](../DT5730_DPP_PSD/build/Build.cpp)。

In [4]:
DEMO_CASES = [
    ("单记录", [1000], [[0]]),
    ("最后一个符合事件", [0, 1000, 80000, 90000], [[0, 1], [2, 3]]),
    ("50 ns 边界", [0, 49999, 50000, 50001], [[0, 1], [2, 3]]),
    ("首记录窗口", [0, 40000, 80000], [[0, 1], [2]]),
]


def run_sample(binary, directory, hits, previous_output=None):
    build_dir = directory / "build"
    build_dir.mkdir(parents=True)
    write_sorted(directory / "sort/run0001_sort.root", hits)
    output = build_dir / "run0001_build.root"
    if previous_output is not None:
        output.write_bytes(previous_output)
    process = subprocess.run([str(binary), "1"], cwd=build_dir,
                             capture_output=True, text=True, timeout=30)
    return process, output


def times_in_events(events):
    return [[hit.timestamp for hit in event] for event in events]


下一单元从 Git 对象读取旧版源码到临时目录，再通过原有 Makefile 编译新旧版本。
它不会切换工作区分支。预期分组直接列在 `DEMO_CASES` 中；当前版本需要通过字段、分组和记录守恒断言。
同一单元还检查超容量和乱序的失败结果，以及既有输出是否保留。

In [5]:
comparison_rows = []
failure_rows = []

with tempfile.TemporaryDirectory(prefix="labr3-notebook-") as temporary:
    work = Path(temporary)
    old_source = work / "baseline_source"
    old_source.mkdir()
    for name in ("Build.cpp", "Build.h", "main.cpp", "set.h", "Makefile"):
        result = checked_run(
            ["git", "show", f"{BASELINE_REV}:{DEMO_VARIANT}/build/{name}"], cwd=REPO)
        (old_source / name).write_text(result.stdout)

    old_binary = work / "build_baseline"
    current_binary = work / "build_current"
    checked_run(["make", "-B", "-C", str(old_source), f"out={old_binary}"])
    checked_run(["make", "-B", "-C", str(REPO / DEMO_VARIANT / "build"),
                 f"out={current_binary}"])

    for index, (label, timestamps, groups) in enumerate(DEMO_CASES):
        hits = hits_at(timestamps)
        expected = [[hits[i] for i in group] for group in groups]
        observed = {}
        for version, binary in (("旧版", old_binary), ("当前版本", current_binary)):
            process, output = run_sample(binary, work / f"case_{index}_{version}", hits)
            assert process.returncode == 0, process.stdout + process.stderr
            _, _, events = read_events(output)
            observed[version] = events
            if version == "当前版本":
                assert events == expected
                assert sum(map(len, events)) == len(hits)
        comparison_rows.append([
            label, len(hits), times_in_events(expected),
            times_in_events(observed["旧版"]), times_in_events(observed["当前版本"]),
            "通过",
        ])

    for index, (label, timestamps, message, preserve) in enumerate([
        ("同窗第 9 条记录", [0] * 9, "multiplicity", False),
        ("窗口内时间倒退", [0, 40000, 30000], "not sorted", False),
        ("失败保留已有输出", [0] + [60000] * 9, "multiplicity", True),
    ]):
        previous = b"existing output sentinel" if preserve else None
        process, output = run_sample(current_binary, work / f"failure_{index}",
                                     hits_at(timestamps), previous)
        assert process.returncode > 0, process.stdout + process.stderr
        assert message in process.stderr.lower()
        if preserve:
            assert output.read_bytes() == previous
        else:
            assert not output.exists()
        assert not list(output.parent.glob("*.partial.*"))
        failure_rows.append([
            label, process.returncode, process.stderr.strip().splitlines()[-1],
            "已有文件保留" if preserve else "未发布最终文件", "已清理",
        ])

ResultTable(["样本", "输入记录数", "预期分组 / ps", "旧版实际输出 / ps",
             "当前实际输出 / ps", "当前版本断言"], comparison_rows)


样本,输入记录数,预期分组 / ps,旧版实际输出 / ps,当前实际输出 / ps,当前版本断言
单记录,1,[[1000]],[],[[1000]],通过
最后一个符合事件,4,"[[0, 1000], [80000, 90000]]","[[0, 1000]]","[[0, 1000], [80000, 90000]]",通过
50 ns 边界,4,"[[0, 49999], [50000, 50001]]","[[0, 49999]]","[[0, 49999], [50000, 50001]]",通过
首记录窗口,3,"[[0, 40000], [80000]]","[[0, 40000]]","[[0, 40000], [80000]]",通过


上表展示的是实际 ROOT 输出。当前版本需要完整保留显式预期的最后一个事件；
旧版输出用于展示缺陷，不参与确定正确答案。能量、板号、通道和时间戳的逐项断言也已经执行。

In [6]:
ResultTable(["样本", "退出码", "诊断", "最终文件", "临时文件"], failure_rows)


样本,退出码,诊断,最终文件,临时文件
同窗第 9 条记录,1,build failed: multiplicity exceeds MAX_MULTI (8) at input entry 8,未发布最终文件,已清理
窗口内时间倒退,1,build failed: timestamps are not sorted at input entry 2,未发布最终文件,已清理
失败保留已有输出,1,build failed: multiplicity exceeds MAX_MULTI (8) at input entry 9,已有文件保留,已清理


接下来复用整个 `tests/` 回归入口，对三套设备配置分别检查。
当前包含 22 项事件构建检查和 4 项 BIN 文件结束检查。

覆盖空输入、末事件、容量边界、时间窗口与顺序、大时间戳、字段和类型、
输入错误、失败文件处理，以及原始转换中的空记录、单记录、多记录和截断末记录。
BIN 检查仅涵盖当前道址能量格式；其截断末记录仍是丢弃后正常退出，后续由 P0-03 完善。

In [7]:
class DeviceCountingResult(unittest.TextTestResult):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.device_cases = 0

    def addSubTest(self, test, subtest, error):
        self.device_cases += 1
        super().addSubTest(test, subtest, error)


suite = unittest.TestLoader().discover(str(REPO / "tests"), pattern="test_*.py")
report = io.StringIO()
test_result = unittest.TextTestRunner(
    stream=report, verbosity=2, resultclass=DeviceCountingResult
).run(suite)
print(report.getvalue())
assert test_result.wasSuccessful(), "回归失败，请查看上方实际日志。"


test_eight_hits (test_build.BuildTests.test_eight_hits) ... ok
test_eight_hits_followed_by_new_window (test_build.BuildTests.test_eight_hits_followed_by_new_window) ... ok
test_empty_tree (test_build.BuildTests.test_empty_tree) ... ok
test_equal_timestamps (test_build.BuildTests.test_equal_timestamps) ... ok
test_failure_preserves_existing_output (test_build.BuildTests.test_failure_preserves_existing_output) ... ok
test_final_coincidence (test_build.BuildTests.test_final_coincidence) ... ok
test_final_single (test_build.BuildTests.test_final_single) ... ok
test_large_timestamps_preserve_picoseconds (test_build.BuildTests.test_large_timestamps_preserve_picoseconds) ... ok
test_missing_branch (test_build.BuildTests.test_missing_branch) ... ok
test_missing_input (test_build.BuildTests.test_missing_input) ... ok
test_missing_tree (test_build.BuildTests.test_missing_tree) ... ok
test_nine_hits_fail (test_build.BuildTests.test_nine_hits_fail) ... ok
test_out_of_order_across_windows (test_bui

In [8]:
ResultTable(["指标", "本次实际运行结果"], [
    ["测试方法数", test_result.testsRun],
    ["设备用例数", test_result.device_cases],
    ["失败数", len(test_result.failures)],
    ["错误数", len(test_result.errors)],
    ["跳过数", len(test_result.skipped)],
    ["整体状态", "通过" if test_result.wasSuccessful() else "失败"],
])


指标,本次实际运行结果
测试方法数,26
设备用例数,78
失败数,0
错误数,0
跳过数,0
整体状态,通过


In [9]:
source_hashes_after = {str(p.relative_to(REPO)): hashlib.sha256(p.read_bytes()).hexdigest()
                       for p in watched_files}
assert source_hashes_before == source_hashes_after, "运行期间源码或配置发生了变化。"
print("验证完成：示例编译和测试没有修改被检查的项目源码及机器路径配置。")


验证完成：示例编译和测试没有修改被检查的项目源码及机器路径配置。


本笔记记录小样本程序行为，完整真实实验数据链和物理结果仍待验证。
截至本次修复，B-01、B-03 和 P0-01 已完成，P0-06 仅完成 `build` 相关部分。

下一项是 P0-02：修复 PSD 能量筛选失败后仍填图的问题，并安全读取超过 2 重的事件。
真实样本来源、刻度适用范围及完整实验分析继续按 [PLAN](../PLAN.md) 推进。
Notebook 的后续重跑反映执行时的工作区状态，具体环境、源码摘要与结果以上面的实际输出为准。